# Lab records in SQL and ANSI C: laser-safety training, journal club, and eyewear bounds

There are two different "ANSI"s in this notebook:
- **ANSI C** (C89) is the language standard. The program compiles with
  `gcc -std=c89 -pedantic -Wall -Wextra -Werror`.
- **ANSI Z136.1** is the US laser-safety standard. The database tracks who has finished training
  and checks eyewear with the same simplified visible-light MPE as `dgs/laser_eyewear.py`. It
  links to each standards body's site and doesn't reproduce any standard's text.

| File | Role |
|---|---|
| `dgs/c/labtrack/schema.sql` | tables, **bounds as CHECK constraints**, and a clearance view |
| `dgs/c/labtrack/seed.sql` | synthetic people, real papers (with DOI URLs), standards bodies by country |
| `dgs/c/labtrack/labtrack.c` | C89 + SQLite: run SQL files, print the report, insert with **prepared statements** |

*Educational only: the institution's Laser Safety Officer decides who works at the bench.*

In [1]:
import sys, sqlite3, tempfile
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
import pandas as pd
from IPython.display import display
from dgs import labtrack as T
from dgs.eventsim import ToolUnavailable
pd.set_option("display.max_colwidth", 70)
DB = Path(tempfile.gettempdir()) / "dgs_labtrack" / "notebook.db"
try:
    T.fresh_db(DB); HAVE_C = True
except ToolUnavailable as err:
    HAVE_C = False; print("skipped (no gcc):", err)

## 1. The report printed by the C program

In [2]:
if HAVE_C:
    print(T.report(DB, "2026-10-29"))

== Laser-safety standards by country ==
Australia / New Zealand  AS/NZS IEC 60825.1     Standards Australia                https://www.standards.org.au
Canada                   CSA E60825-1           CSA Group                          https://www.csagroup.org
European Union           EN 60825-1             CENELEC                            https://www.cenelec.eu
International            IEC 60825-1            IEC                                https://www.iec.ch
Japan                    JIS C 6802             JSA / JISC                         https://www.jisc.go.jp
United Kingdom           BS EN 60825-1          BSI                                https://www.bsigroup.com
United States            ANSI Z136.1            ANSI / Laser Institute of America  https://www.lia.org
United States            ANSI Z136 series (accreditation) ANSI                               https://www.ansi.org

== Journal club from 2026-10-29 ==
2026-11-05  Blake (grad)     [photonics] Dispersive Fourier trans

## 2. SQL you should be able to write

These are the queries the C program runs, written in SQL: a JOIN, a GROUP BY with a
conditional aggregate, and a correlated subquery.

In [3]:
if HAVE_C:
    con = sqlite3.connect(DB)
    queries = {
        "standards per country (GROUP BY)": "SELECT country, COUNT(*) AS n, group_concat(document, '; ') AS documents FROM standard GROUP BY country",
        "journal club with presenter (JOIN)": "SELECT j.held_on, m.name AS presenter, j.topic, j.doi_url FROM journal_club j JOIN member m ON m.id = j.presenter_id ORDER BY j.held_on",
        "attendance per session (subquery)": "SELECT j.title, (SELECT COUNT(*) FROM attendance a WHERE a.session_id = j.id) AS attended FROM journal_club j",
        "clearance view": "SELECT * FROM clearance",
    }
    for title, q in queries.items():
        print(title); print("   ", q); display(pd.read_sql(q, con))

standards per country (GROUP BY)
    SELECT country, COUNT(*) AS n, group_concat(document, '; ') AS documents FROM standard GROUP BY country


,country,n,documents
0,Australia / New Zealand,1,AS/NZS IEC 60825.1
1,Canada,1,CSA E60825-1
2,European Union,1,EN 60825-1
3,International,1,IEC 60825-1
4,Japan,1,JIS C 6802
5,United Kingdom,1,BS EN 60825-1
6,United States,2,ANSI Z136.1; ANSI Z136 series (accreditation)


journal club with presenter (JOIN)
    SELECT j.held_on, m.name AS presenter, j.topic, j.doi_url FROM journal_club j JOIN member m ON m.id = j.presenter_id ORDER BY j.held_on


,held_on,presenter,topic,doi_url
0,2026-11-05,Blake (grad),photonics,https://doi.org/10.1038/nphoton.2013.131
1,2026-11-12,Avery (student),phase retrieval,https://doi.org/10.1364/AO.21.002758
2,2026-11-19,Casey (staff),quantum information,https://doi.org/10.1103/PhysRevA.76.042319
3,2026-12-03,Devon (student),computing,https://doi.org/10.1007/BF02650179


attendance per session (subquery)
    SELECT j.title, (SELECT COUNT(*) FROM attendance a WHERE a.session_id = j.id) AS attended FROM journal_club j


,title,attended
0,Dispersive Fourier transformation for fast continuous single-shot ...,3
1,"Phase retrieval algorithms: a comparison (Fienup, 1982)",2
2,Charge-insensitive qubit design derived from the Cooper pair box (...,0
3,"Simulating physics with computers (Feynman, 1982)",0


clearance view
    SELECT * FROM clearance


,member_id,name,role,laser_safety_expires,site_expires
0,1,Avery (student),student,2027-09-02,2027-09-10
1,2,Blake (grad),grad,2027-10-20,2027-10-21
2,3,Casey (staff),staff,2027-01-05,None
3,4,Devon (student),student,None,None


## 3. Bounds: the database refuses impossible data

Each row below breaks one CHECK or foreign-key constraint. The database rejects it, whichever
program tries to write it: Python here, C in the next section.

In [4]:
bad = [
    ("wavelength below 180 nm", "INSERT INTO laser (name, wavelength_nm, power_w, beam_diameter_mm, hazard_class) VALUES ('uv', 100, 1, 1, '4')"),
    ("negative power", "INSERT INTO laser (name, wavelength_nm, power_w, beam_diameter_mm, hazard_class) VALUES ('n', 532, -1, 1, '2')"),
    ("laser class 5 doesn't exist", "INSERT INTO laser (name, wavelength_nm, power_w, beam_diameter_mm, hazard_class) VALUES ('c', 532, 1, 1, '5')"),
    ("OD above 10", "INSERT INTO eyewear (label, laser_id, od, lo_nm, hi_nm) VALUES ('x', 1, 11, 500, 560)"),
    ("eyewear for a laser that doesn't exist", "INSERT INTO eyewear (label, laser_id, od, lo_nm, hi_nm) VALUES ('x', 99, 3, 500, 560)"),
    ("non-https URL", "INSERT INTO standard (country, body, document, scope, url) VALUES ('X','Y','Z','s','http://x')"),
    ("training expires before completion", "INSERT INTO training (member_id, topic, completed, expires) VALUES (1, 'laser safety', '2026-10-01', '2026-09-01')"),
]
if HAVE_C:
    con.execute("PRAGMA foreign_keys = ON")
    rows = []
    for why, q in bad:
        try:
            con.execute(q); rows.append((why, "ACCEPTED (bug!)"))
        except sqlite3.IntegrityError as err:
            rows.append((why, f"rejected: {err}"))
    display(pd.DataFrame(rows, columns=["bad row", "result"]))

,bad row,result
0,wavelength below 180 nm,rejected: CHECK constraint failed: wavelength_nm BETWEEN 180 AND 1...
1,negative power,rejected: CHECK constraint failed: power_w > 0 AND power_w <= 1000
2,laser class 5 doesn't exist,"rejected: CHECK constraint failed: hazard_class IN ('1', '1M', '2'..."
3,OD above 10,rejected: CHECK constraint failed: od BETWEEN 0 AND 10
4,eyewear for a laser that doesn't exist,rejected: FOREIGN KEY constraint failed
5,non-https URL,rejected: CHECK constraint failed: url LIKE 'https://%'
6,training expires before completion,rejected: CHECK constraint failed: expires > completed


## 4. ANSI C: prepared statements stop SQL injection

`add-laser` binds each argument with `sqlite3_bind_*` instead of pasting text into the SQL.
A name like `x'); DROP TABLE laser; --` is stored as ordinary text, and the bounds still apply.

In [5]:
if HAVE_C:
    print(T.run(DB, "add-laser", "uv", 100, 1, 1, "4")[2])
    rc, out, err = T.run(DB, "add-laser", "x'); DROP TABLE laser; --", 532, 0.001, 1, "2")
    print(out); display(pd.read_sql("SELECT id, name, wavelength_nm, hazard_class FROM laser", con))

rejected: database is locked




,id,name,wavelength_nm,hazard_class
0,1,green alignment pointer,532.0,3R
1,2,HeNe 633 bench laser,633.0,3B
2,3,mode-locked fiber laser 1550,1550.0,3B


## 5. The eyewear check, cross-checked against Python

The C program computes the exit-beam irradiance E = 4P/(πd²) and the 0.25 s visible MPE, then
OD_needed = log₁₀(E/MPE). The same numbers come from `dgs/laser_eyewear.py`. A 5 mW green pointer
with a 1 mm beam needs about **OD 2.4** at the aperture, so "OD 2" goggles **fail** for direct
viewing. A HeNe with green goggles fails for a different reason: the goggles aren't rated for
that wavelength.

In [6]:
if HAVE_C:
    from dgs import laser_eyewear as LS
    rows = T.eyewear_rows(T.report(DB, "2026-10-29"))
    mpe = LS.mpe_visible(LS.BLINK_S)
    for r in rows:
        if r["need_od"] is not None:
            r["python need_od"] = round(LS.required_od(r["E_W_cm2"], mpe), 2)
    display(pd.DataFrame(rows))
    con.close()

,label,laser,E_W_cm2,need_od,od,verdict,python need_od
0,OD2 green goggles,green alignment pointer,0.6366,2.40,2.0,FAIL: optical density too low,2.40
1,OD2 green goggles (wrong use),HeNe 633 bench laser,3.9790,NaN,2.0,FAIL: laser wavelength outside the eyewear's rated band,NaN
2,OD4 red goggles,HeNe 633 bench laser,3.9790,3.19,4.0,OK,3.19
3,OD5 IR goggles,mode-locked fiber laser 1550,1.5920,NaN,5.0,UNCHECKED: outside the simplified 400-700 nm MPE; ask the LSO,NaN
